# SB 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from tabicl import TabICLForecaster

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\tabicl_5min_SB"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288
CONTEXT_LEN = 288 * 30

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample("5min").mean()
weather_5min = weather_df.resample("5min").mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.dropna(subset=[TARGET_COL] + WEATHER_COLS).reset_index(drop=True)

print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())
print(df[[TARGET_COL] + WEATHER_COLS].head())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# MODEL
# ============================================================
forecaster = TabICLForecaster()

# ============================================================
# HELPERS
# ============================================================
def find_prediction_column(pred_df):
    possible_cols = ["prediction", "predictions", "mean", "median", "0.5", 0.5]

    for col in possible_cols:
        if col in pred_df.columns:
            return col

    numeric_cols = pred_df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["timestamp"]]

    if len(numeric_cols) > 0:
        return numeric_cols[0]

    raise ValueError(
        f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}"
    )

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast_tabicl_sb(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_actual_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None

    if len(future_actual_df) != PRED_LEN:
        print(
            f"Skipping {season}-{day_name}: expected {PRED_LEN}, "
            f"got {len(future_actual_df)}"
        )
        return None

    # SB input: historical target + raw weather only
    train_df = history_df[["timestamp", TARGET_COL] + WEATHER_COLS].copy()
    train_df = train_df.rename(columns={TARGET_COL: "target"})

    train_df["item_id"] = "building_consumption"

    train_df = train_df[
        ["item_id", "timestamp", "target"] + WEATHER_COLS
    ]

    # No future_df is passed.
    # This matches your Chronos SB setup.
    pred_df = forecaster.predict_df(
        train_df,
        prediction_length=PRED_LEN,
    )

    print(pred_df.head())
    print(pred_df.columns.tolist())

    pred_col = find_prediction_column(pred_df)
    pred_values = pred_df[pred_col].values[:PRED_LEN]

    out = future_actual_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name

    return out

# ============================================================
# RUN ALL FORECASTS
# ============================================================
start_time = time.time()

all_results = []

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running TabiCL SB - {season} - {day_name}: {test_start}")

        res = run_one_forecast_tabicl_sb(season, day_name, test_start)

        if res is not None:
            all_results.append(res)

            day_path = OUTPUT_DIR / f"tabicl_5min_SB_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            print(f"Saved: {day_path}")

if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

results_df = pd.concat(all_results, ignore_index=True)

combined_path = OUTPUT_DIR / "tabicl_5min_SB_all_results.csv"
results_df.to_csv(combined_path, index=False)

elapsed = time.time() - start_time

print("=" * 80)
print(f"Saved combined results: {combined_path}")
print(f"Elapsed time: {elapsed:.2f} seconds")
print(results_df.head())

c:\Users\Dimitrios\anaconda3\envs\tabicl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data range: 2025-01-01 00:00:00 to 2025-12-31 23:55:00
   avg_power  Temp__C  Hum__  AvgWindSpeed_M_s  Rain_Mm  SolarRad_W_m_2
0  2954.2428     -3.4   94.6               0.1      0.0             0.0
1  2642.1732     -3.4   94.6               0.2      0.0             0.0
2  2646.2412     -3.4   94.6               0.1      0.0             0.0
3  2685.5580     -3.4   94.6               0.1      0.0             0.0
4  2665.8096     -3.4   94.6               0.1      0.0             0.0
Running TabiCL SB - winter - day1: 2025-01-31 00:00:00


Predicting time series: 100%|██████████| 1/1 [01:20<00:00, 80.88s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2844.597656  2368.568359   
                     2025-01-31 00:05:00  2837.366699  2364.822021   
                     2025-01-31 00:10:00  2837.853027  2362.253906   
                     2025-01-31 00:15:00  2843.075684  2358.536377   
                     2025-01-31 00:20:00  2844.408691  2354.902344   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2398.119873  2422.825928   
                     2025-01-31 00:05:00  2393.940186  2417.772217   
                     2025-01-31 00:10:00  2390.920166  2414.316895   
                     2025-01-31 00:15:00  2386.791504  2409.838867   
                     2025-01-31 00:20:00  2382.607666  2405.303223   

                  

Predicting time series: 100%|██████████| 1/1 [01:12<00:00, 72.79s/it]


...

                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  7146.065430  2572.209229   
                     2025-07-17 00:05:00  7144.788574  2570.455078   
                     2025-07-17 00:10:00  7174.800293  2545.986328   
                     2025-07-17 00:15:00  7226.994141  2512.854004   
                     2025-07-17 00:20:00  7289.166992  2472.617676   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  3488.141846  4120.773438   
                     2025-07-17 00:05:00  3488.238525  4118.557617   
                     2025-07-17 00:10:00  3472.157471  4110.324219   
                     2025-07-17 00:15:00  3450.340576  4101.677734   
                     2025-07-17 00:20:00  3420.748779  4091.726562   

                  

Predicting time series: 100%|██████████| 1/1 [01:09<00:00, 69.72s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  4510.270508  2370.570801   
                     2025-08-08 00:05:00  4526.028809  2370.303955   
                     2025-08-08 00:10:00  4535.745605  2360.819824   
                     2025-08-08 00:15:00  4547.128906  2351.031738   
                     2025-08-08 00:20:00  4552.215820  2342.420898   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  2790.154297  3101.368164   
                     2025-08-08 00:05:00  2790.948730  3103.807617   
                     2025-08-08 00:10:00  2783.701172  3101.586670   
                     2025-08-08 00:15:00  2777.250244  3100.126221   
                     2025-08-08 00:20:00  2771.819092  3098.970703   

                  

Predicting time series: 100%|██████████| 1/1 [01:08<00:00, 68.69s/it]

                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2735.582520  2123.842285   
                     2025-08-31 00:05:00  2732.994873  2123.227295   
                     2025-08-31 00:10:00  2731.281006  2122.497314   
                     2025-08-31 00:15:00  2730.899658  2121.686523   
                     2025-08-31 00:20:00  2730.689453  2120.642090   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2229.976318  2301.489014   
                     2025-08-31 00:05:00  2228.629639  2299.034912   
                     2025-08-31 00:10:00  2227.678223  2297.187256   
                     2025-08-31 00:15:00  2227.047119  2295.911865   
                     2025-08-31 00:20:00  2226.361328  2294.701416   

                  

# AF

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from tabicl import TabICLForecaster

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\tabicl_5min_engineered"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

BASE_WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288          # 24h @ 5-min
CONTEXT_LEN = 288 * 30  # 30 days

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample("5min").mean()
weather_5min = weather_df.resample("5min").mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# ENGINEERED FEATURES
# ============================================================
def add_engineered_features(data, target_col):
    data = data.copy()
    ts = data["timestamp"]

    # Time features
    data["hour"] = ts.dt.hour
    data["minute"] = ts.dt.minute
    data["dayofweek"] = ts.dt.dayofweek
    data["dayofyear"] = ts.dt.dayofyear
    data["month"] = ts.dt.month
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

    # Fourier features
    steps_per_day = 288
    steps_per_week = 288 * 7

    step_of_day = data["hour"] * 12 + data["minute"] // 5
    step_of_week = data["dayofweek"] * steps_per_day + step_of_day

    for k in [1, 2, 3]:
        data[f"sin_day_{k}"] = np.sin(2 * np.pi * k * step_of_day / steps_per_day)
        data[f"cos_day_{k}"] = np.cos(2 * np.pi * k * step_of_day / steps_per_day)

    for k in [1, 2]:
        data[f"sin_week_{k}"] = np.sin(2 * np.pi * k * step_of_week / steps_per_week)
        data[f"cos_week_{k}"] = np.cos(2 * np.pi * k * step_of_week / steps_per_week)

    # Load lag features
    lag_steps = [
        1,
        2,
        3,
        6,
        12,
        24,
        288,
        288 * 2,
        288 * 7,
    ]

    for lag in lag_steps:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    # Rolling historical load features
    rolling_windows = [
        3,
        12,
        288,
        288 * 7,
    ]

    shifted = data[target_col].shift(1)

    for window in rolling_windows:
        data[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
        data[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()

    # Weather-derived features
    if "Temp__C" in data.columns:
        data["temp_sq"] = data["Temp__C"] ** 2
        data["heating_degree_18"] = np.maximum(18 - data["Temp__C"], 0)
        data["cooling_degree_22"] = np.maximum(data["Temp__C"] - 22, 0)

    if "SolarRad_W_m_2" in data.columns:
        data["is_daylight"] = (data["SolarRad_W_m_2"] > 10).astype(int)

    return data


df = add_engineered_features(df, TARGET_COL)

# ============================================================
# FEATURE LIST
# ============================================================
ENGINEERED_COLS = [
    "hour",
    "minute",
    "dayofweek",
    "dayofyear",
    "month",
    "is_weekend",

    "sin_day_1", "cos_day_1",
    "sin_day_2", "cos_day_2",
    "sin_day_3", "cos_day_3",

    "sin_week_1", "cos_week_1",
    "sin_week_2", "cos_week_2",

    f"{TARGET_COL}_lag_1",
    f"{TARGET_COL}_lag_2",
    f"{TARGET_COL}_lag_3",
    f"{TARGET_COL}_lag_6",
    f"{TARGET_COL}_lag_12",
    f"{TARGET_COL}_lag_24",
    f"{TARGET_COL}_lag_288",
    f"{TARGET_COL}_lag_576",
    f"{TARGET_COL}_lag_2016",

    f"{TARGET_COL}_roll_mean_3",
    f"{TARGET_COL}_roll_std_3",
    f"{TARGET_COL}_roll_mean_12",
    f"{TARGET_COL}_roll_std_12",
    f"{TARGET_COL}_roll_mean_288",
    f"{TARGET_COL}_roll_std_288",
    f"{TARGET_COL}_roll_mean_2016",
    f"{TARGET_COL}_roll_std_2016",

    "temp_sq",
    "heating_degree_18",
    "cooling_degree_22",
    "is_daylight",
]

FEATURE_COLS = BASE_WEATHER_COLS + ENGINEERED_COLS

df = df.dropna(subset=[TARGET_COL] + FEATURE_COLS).reset_index(drop=True)

print("Final feature columns:")
print(FEATURE_COLS)
print("Number of features:", len(FEATURE_COLS))
print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())

# ============================================================
# LOAD FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# LOAD TABICL
# ============================================================
forecaster = TabICLForecaster()

# ============================================================
# HELPER
# ============================================================
def find_prediction_column(pred_df):
    possible_cols = [
        "prediction",
        "predictions",
        "mean",
        "median",
        "0.5",
        0.5,
        "target",
    ]

    for col in possible_cols:
        if col in pred_df.columns:
            return col

    numeric_cols = pred_df.select_dtypes(include=[np.number]).columns.tolist()

    numeric_cols = [
        c for c in numeric_cols
        if c not in ["timestamp"]
    ]

    if len(numeric_cols) > 0:
        return numeric_cols[0]

    raise ValueError(
        f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}"
    )

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None

    # Historical target + historical engineered features
    tabicl_df = history_df[["timestamp", TARGET_COL] + FEATURE_COLS].copy()
    tabicl_df = tabicl_df.rename(columns={TARGET_COL: "target"})

    tabicl_df["item_id"] = "building_consumption"

    tabicl_df = tabicl_df[
        ["item_id", "timestamp", "target"] + FEATURE_COLS
    ]

    # IMPORTANT:
    # No future_df is passed here.
    # This matches your Chronos setup: only historical target + historical features.
    pred_df = forecaster.predict_df(
        tabicl_df,
        prediction_length=PRED_LEN,
    )

    print(pred_df.head())
    print(pred_df.columns.tolist())

    pred_col = find_prediction_column(pred_df)
    pred_values = pred_df[pred_col].values[:PRED_LEN]

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name

    return out

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running TabiCL engineered - {season} - {day_name}: {test_start}")

        res = run_one_forecast(season, day_name, test_start)

        if res is not None:
            all_results.append(res)

            day_path = OUTPUT_DIR / f"tabicl_5min_AF_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            print(f"Saved: {day_path}")

if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

results_df = pd.concat(all_results, ignore_index=True)

all_path = OUTPUT_DIR / "tabicl_5min_AF_predictions.csv"
results_df.to_csv(all_path, index=False)

print(f"Saved combined predictions: {all_path}")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")

Final feature columns:
['Temp__C', 'Hum__', 'AvgWindSpeed_M_s', 'Rain_Mm', 'SolarRad_W_m_2', 'hour', 'minute', 'dayofweek', 'dayofyear', 'month', 'is_weekend', 'sin_day_1', 'cos_day_1', 'sin_day_2', 'cos_day_2', 'sin_day_3', 'cos_day_3', 'sin_week_1', 'cos_week_1', 'sin_week_2', 'cos_week_2', 'avg_power_lag_1', 'avg_power_lag_2', 'avg_power_lag_3', 'avg_power_lag_6', 'avg_power_lag_12', 'avg_power_lag_24', 'avg_power_lag_288', 'avg_power_lag_576', 'avg_power_lag_2016', 'avg_power_roll_mean_3', 'avg_power_roll_std_3', 'avg_power_roll_mean_12', 'avg_power_roll_std_12', 'avg_power_roll_mean_288', 'avg_power_roll_std_288', 'avg_power_roll_mean_2016', 'avg_power_roll_std_2016', 'temp_sq', 'heating_degree_18', 'cooling_degree_22', 'is_daylight']
Number of features: 42
Data range: 2025-01-08 00:00:00 to 2025-12-31 23:55:00
Running TabiCL engineered - winter - day1: 2025-01-31 00:00:00


Predicting time series: 100%|██████████| 1/1 [00:55<00:00, 55.73s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2828.602051  2391.141357   
                     2025-01-31 00:05:00  2819.485352  2386.819824   
                     2025-01-31 00:10:00  2818.965576  2383.406982   
                     2025-01-31 00:15:00  2824.997314  2379.065674   
                     2025-01-31 00:20:00  2822.512207  2374.582520   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2418.610840  2438.500244   
                     2025-01-31 00:05:00  2413.788330  2433.183838   
                     2025-01-31 00:10:00  2410.037598  2428.999512   
                     2025-01-31 00:15:00  2405.457764  2424.266602   
                     2025-01-31 00:20:00  2400.451660  2418.936523   

                  

Predicting time series: 100%|██████████| 1/1 [01:05<00:00, 65.56s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-02-15 00:00:00  2790.829102  2386.102051   
                     2025-02-15 00:05:00  2823.225830  2387.878662   
                     2025-02-15 00:10:00  2842.538574  2387.947754   
                     2025-02-15 00:15:00  2852.337891  2389.028809   
                     2025-02-15 00:20:00  2847.596680  2390.529785   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-02-15 00:00:00  2441.437012  2479.596680   
                     2025-02-15 00:05:00  2443.124512  2483.190430   
                     2025-02-15 00:10:00  2441.587646  2481.570312   
                     2025-02-15 00:15:00  2440.800049  2480.200928   
                     2025-02-15 00:20:00  2440.952637  2480.178711   

                  

...

                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  6970.562988  2682.639893   
                     2025-07-17 00:05:00  6963.860352  2663.129395   
                     2025-07-17 00:10:00  6998.870605  2626.306152   
                     2025-07-17 00:15:00  7062.632324  2586.283691   
                     2025-07-17 00:20:00  7137.463867  2545.043457   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  3587.505859  4183.688965   
                     2025-07-17 00:05:00  3569.259766  4170.597656   
                     2025-07-17 00:10:00  3541.469727  4155.972168   
                     2025-07-17 00:15:00  3514.193604  4145.277832   
                     2025-07-17 00:20:00  3484.684326  4136.643066   

                  

Predicting time series: 100%|██████████| 1/1 [01:08<00:00, 68.15s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  4547.494629  2238.516357   
                     2025-08-08 00:05:00  4556.206055  2238.433350   
                     2025-08-08 00:10:00  4559.377930  2232.340576   
                     2025-08-08 00:15:00  4563.788086  2227.084229   
                     2025-08-08 00:20:00  4562.586914  2222.557373   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  2674.558594  3024.015381   
                     2025-08-08 00:05:00  2674.255615  3024.289795   
                     2025-08-08 00:10:00  2668.567139  3021.636963   
                     2025-08-08 00:15:00  2664.458008  3021.056152   
                     2025-08-08 00:20:00  2660.720703  3020.008545   

                  

Predicting time series: 100%|██████████| 1/1 [01:07<00:00, 67.68s/it]

                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2805.875732  2149.622070   
                     2025-08-31 00:05:00  2801.495850  2148.943359   
                     2025-08-31 00:10:00  2797.872070  2148.468750   
                     2025-08-31 00:15:00  2796.235596  2147.950439   
                     2025-08-31 00:20:00  2794.898682  2146.889648   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2259.388672  2330.942383   
                     2025-08-31 00:05:00  2257.177002  2327.437500   
                     2025-08-31 00:10:00  2255.825439  2325.042236   
                     2025-08-31 00:15:00  2255.078857  2323.574463   
                     2025-08-31 00:20:00  2254.012939  2322.024902   

                  

# RFFS

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from tabicl import TabICLForecaster
from sklearn.ensemble import RandomForestRegressor

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\tabicl_5min_engineered_RF95"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

BASE_WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288
CONTEXT_LEN = 288 * 30

RF_IMPORTANCE_COVERAGE = 0.95

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample("5min").mean()
weather_5min = weather_df.resample("5min").mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# ENGINEERED FEATURES
# ============================================================
def add_engineered_features(data, target_col):
    data = data.copy()
    ts = data["timestamp"]

    data["hour"] = ts.dt.hour
    data["minute"] = ts.dt.minute
    data["dayofweek"] = ts.dt.dayofweek
    data["dayofyear"] = ts.dt.dayofyear
    data["month"] = ts.dt.month
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

    steps_per_day = 288
    steps_per_week = 288 * 7

    step_of_day = data["hour"] * 12 + data["minute"] // 5
    step_of_week = data["dayofweek"] * steps_per_day + step_of_day

    for k in [1, 2, 3]:
        data[f"sin_day_{k}"] = np.sin(2 * np.pi * k * step_of_day / steps_per_day)
        data[f"cos_day_{k}"] = np.cos(2 * np.pi * k * step_of_day / steps_per_day)

    for k in [1, 2]:
        data[f"sin_week_{k}"] = np.sin(2 * np.pi * k * step_of_week / steps_per_week)
        data[f"cos_week_{k}"] = np.cos(2 * np.pi * k * step_of_week / steps_per_week)

    lag_steps = [
        1, 2, 3, 6, 12, 24,
        288,
        288 * 2,
        288 * 7,
    ]

    for lag in lag_steps:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    rolling_windows = [3, 12, 288, 288 * 7]

    shifted = data[target_col].shift(1)

    for window in rolling_windows:
        data[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
        data[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()

    data["temp_sq"] = data["Temp__C"] ** 2
    data["heating_degree_18"] = np.maximum(18 - data["Temp__C"], 0)
    data["cooling_degree_22"] = np.maximum(data["Temp__C"] - 22, 0)
    data["is_daylight"] = (data["SolarRad_W_m_2"] > 10).astype(int)

    return data


df = add_engineered_features(df, TARGET_COL)

ENGINEERED_COLS = [
    "hour", "minute", "dayofweek", "dayofyear", "month", "is_weekend",

    "sin_day_1", "cos_day_1",
    "sin_day_2", "cos_day_2",
    "sin_day_3", "cos_day_3",

    "sin_week_1", "cos_week_1",
    "sin_week_2", "cos_week_2",

    f"{TARGET_COL}_lag_1",
    f"{TARGET_COL}_lag_2",
    f"{TARGET_COL}_lag_3",
    f"{TARGET_COL}_lag_6",
    f"{TARGET_COL}_lag_12",
    f"{TARGET_COL}_lag_24",
    f"{TARGET_COL}_lag_288",
    f"{TARGET_COL}_lag_576",
    f"{TARGET_COL}_lag_2016",

    f"{TARGET_COL}_roll_mean_3",
    f"{TARGET_COL}_roll_std_3",
    f"{TARGET_COL}_roll_mean_12",
    f"{TARGET_COL}_roll_std_12",
    f"{TARGET_COL}_roll_mean_288",
    f"{TARGET_COL}_roll_std_288",
    f"{TARGET_COL}_roll_mean_2016",
    f"{TARGET_COL}_roll_std_2016",

    "temp_sq",
    "heating_degree_18",
    "cooling_degree_22",
    "is_daylight",
]

ALL_FEATURE_COLS = BASE_WEATHER_COLS + ENGINEERED_COLS

df = df.dropna(subset=[TARGET_COL] + ALL_FEATURE_COLS).reset_index(drop=True)

print("Total candidate features:", len(ALL_FEATURE_COLS))
print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# TABICL
# ============================================================
forecaster = TabICLForecaster()

# ============================================================
# RANDOM FOREST 95% IMPORTANCE SELECTION
# ============================================================
def select_rf_95_features(train_df, feature_cols, target_col, season, day_name):
    X = train_df[feature_cols].copy()
    y = train_df[target_col].copy()

    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )

    rf.fit(X, y)

    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": rf.feature_importances_,
    })

    importance_df = importance_df.sort_values(
        "importance", ascending=False
    ).reset_index(drop=True)

    total_importance = importance_df["importance"].sum()

    if total_importance == 0:
        selected_features = feature_cols
        importance_df["importance_share"] = 0.0
        importance_df["cumulative_importance"] = 0.0
    else:
        importance_df["importance_share"] = (
            importance_df["importance"] / total_importance
        )

        importance_df["cumulative_importance"] = (
            importance_df["importance_share"].cumsum()
        )

        selected_features = importance_df.loc[
            importance_df["cumulative_importance"] <= RF_IMPORTANCE_COVERAGE,
            "feature"
        ].tolist()

        # include the first feature that crosses 95%
        if len(selected_features) < len(importance_df):
            crossing_feature = importance_df.iloc[len(selected_features)]["feature"]
            selected_features.append(crossing_feature)

    if len(selected_features) == 0:
        selected_features = importance_df.head(1)["feature"].tolist()

    importance_df["selected"] = importance_df["feature"].isin(selected_features)
    importance_df["season"] = season
    importance_df["day"] = day_name

    return selected_features, importance_df

# ============================================================
# HELPER: FIND PREDICTION COLUMN
# ============================================================
def find_prediction_column(pred_df):
    possible_cols = [
        "prediction",
        "predictions",
        "mean",
        "median",
        "0.5",
        0.5,
        "target",
    ]

    for col in possible_cols:
        if col in pred_df.columns:
            return col

    numeric_cols = pred_df.select_dtypes(include=[np.number]).columns.tolist()

    numeric_cols = [
        c for c in numeric_cols
        if c not in ["timestamp"]
    ]

    if len(numeric_cols) > 0:
        return numeric_cols[0]

    raise ValueError(
        f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}"
    )

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    train_for_selection_df = df[df["timestamp"] < test_start].copy()

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(train_for_selection_df) == 0:
        print(f"Skipping {season}-{day_name}: no training data before {test_start}")
        return None, None, None

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None, None, None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None, None, None

    selected_features, importance_df = select_rf_95_features(
        train_df=train_for_selection_df,
        feature_cols=ALL_FEATURE_COLS,
        target_col=TARGET_COL,
        season=season,
        day_name=day_name,
    )

    print(f"Selected {len(selected_features)} / {len(ALL_FEATURE_COLS)} features")
    print(selected_features)

    tabicl_df = history_df[["timestamp", TARGET_COL] + selected_features].copy()
    tabicl_df = tabicl_df.rename(columns={TARGET_COL: "target"})

    tabicl_df["item_id"] = "building_consumption"

    tabicl_df = tabicl_df[
        ["item_id", "timestamp", "target"] + selected_features
    ]

    # IMPORTANT:
    # No future_df is passed.
    # TabiCL receives only historical target + RF-selected historical features.
    pred_df = forecaster.predict_df(
        tabicl_df,
        prediction_length=PRED_LEN,
    )

    print(pred_df.head())
    print(pred_df.columns.tolist())

    pred_col = find_prediction_column(pred_df)
    pred_values = pred_df[pred_col].values[:PRED_LEN]

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name
    out["n_selected_features"] = len(selected_features)

    selected_features_df = pd.DataFrame({
        "season": season,
        "day": day_name,
        "selected_feature": selected_features,
    })

    return out, importance_df, selected_features_df

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []
all_importances = []
all_selected_features = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running TabiCL RF95 - {season} - {day_name}: {test_start}")

        res, importance_df, selected_features_df = run_one_forecast(
            season, day_name, test_start
        )

        if res is not None:
            all_results.append(res)
            all_importances.append(importance_df)
            all_selected_features.append(selected_features_df)

            day_path = OUTPUT_DIR / f"tabicl_5min_RFFS_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            imp_path = OUTPUT_DIR / f"RF95_importance_{season}_{day_name}.csv"
            importance_df.to_csv(imp_path, index=False)

            selected_path = OUTPUT_DIR / f"RF95_selected_features_{season}_{day_name}.csv"
            selected_features_df.to_csv(selected_path, index=False)

            print(f"Saved prediction: {day_path}")
            print(f"Saved importance: {imp_path}")
            print(f"Saved selected features: {selected_path}")

# ============================================================
# SAVE COMBINED FILES
# ============================================================
if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

results_df = pd.concat(all_results, ignore_index=True)
importance_all_df = pd.concat(all_importances, ignore_index=True)
selected_all_df = pd.concat(all_selected_features, ignore_index=True)

results_df.to_csv(
    OUTPUT_DIR / "tabicl_5min_RF95_ALL_predictions.csv",
    index=False
)

importance_all_df.to_csv(
    OUTPUT_DIR / "RF95_ALL_importances.csv",
    index=False
)

selected_all_df.to_csv(
    OUTPUT_DIR / "RF95_ALL_selected_features.csv",
    index=False
)

print("Done.")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
print(f"Output folder: {OUTPUT_DIR}")

Total candidate features: 42
Data range: 2025-01-08 00:00:00 to 2025-12-31 23:55:00
Running TabiCL RF95 - winter - day1: 2025-01-31 00:00:00
Selected 14 / 42 features
['avg_power_lag_1', 'avg_power_roll_mean_3', 'avg_power_lag_2', 'avg_power_roll_mean_12', 'avg_power_lag_2016', 'avg_power_roll_mean_288', 'avg_power_lag_3', 'avg_power_lag_6', 'sin_day_1', 'avg_power_lag_288', 'avg_power_lag_576', 'avg_power_roll_std_12', 'avg_power_roll_std_3', 'avg_power_roll_mean_2016']


Predicting time series: 100%|██████████| 1/1 [01:09<00:00, 69.26s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2809.540283  2388.081787   
                     2025-01-31 00:05:00  2803.230469  2384.605713   
                     2025-01-31 00:10:00  2804.303711  2382.300049   
                     2025-01-31 00:15:00  2811.358154  2378.728516   
                     2025-01-31 00:20:00  2814.275879  2375.420898   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2411.010254  2433.599609   
                     2025-01-31 00:05:00  2407.307617  2429.385742   
                     2025-01-31 00:10:00  2404.657227  2426.530518   
                     2025-01-31 00:15:00  2400.739258  2422.569580   
                     2025-01-31 00:20:00  2397.111328  2418.840820   

                  

Predicting time series: 100%|██████████| 1/1 [01:06<00:00, 66.01s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-02-15 00:00:00  2827.658936  2388.806152   
                     2025-02-15 00:05:00  2853.020752  2392.259766   
                     2025-02-15 00:10:00  2864.350830  2392.917725   
                     2025-02-15 00:15:00  2865.357666  2394.153076   
                     2025-02-15 00:20:00  2850.775146  2395.269287   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-02-15 00:00:00  2444.962891  2488.658936   
                     2025-02-15 00:05:00  2448.549316  2493.793945   
                     2025-02-15 00:10:00  2448.030029  2492.902588   
                     2025-02-15 00:15:00  2447.479492  2491.271484   
                     2025-02-15 00:20:00  2446.805664  2489.671387   

                  

...

                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  6919.363281  2766.905273   
                     2025-07-17 00:05:00  6912.711426  2759.213379   
                     2025-07-17 00:10:00  6949.357422  2727.400391   
                     2025-07-17 00:15:00  7010.851562  2689.833496   
                     2025-07-17 00:20:00  7076.307617  2650.461914   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  3755.735107  4320.501465   
                     2025-07-17 00:05:00  3743.720459  4310.384766   
                     2025-07-17 00:10:00  3715.713135  4294.224609   
                     2025-07-17 00:15:00  3685.946533  4280.386719   
                     2025-07-17 00:20:00  3652.465576  4264.183594   

                  

Predicting time series: 100%|██████████| 1/1 [01:13<00:00, 73.05s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  4635.485840  2325.772705   
                     2025-08-08 00:05:00  4648.700684  2324.493408   
                     2025-08-08 00:10:00  4651.925781  2314.358643   
                     2025-08-08 00:15:00  4660.156738  2303.060547   
                     2025-08-08 00:20:00  4655.585938  2295.767334   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  2767.690430  3095.927979   
                     2025-08-08 00:05:00  2768.949707  3097.568115   
                     2025-08-08 00:10:00  2760.736084  3092.708008   
                     2025-08-08 00:15:00  2752.808105  3088.900391   
                     2025-08-08 00:20:00  2747.698242  3085.947754   

                  

Predicting time series: 100%|██████████| 1/1 [01:08<00:00, 68.54s/it]

                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2762.120850  2151.958008   
                     2025-08-31 00:05:00  2758.373535  2151.118164   
                     2025-08-31 00:10:00  2756.295166  2150.350098   
                     2025-08-31 00:15:00  2756.379395  2149.817139   
                     2025-08-31 00:20:00  2756.108643  2148.814941   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2251.408203  2318.923828   
                     2025-08-31 00:05:00  2249.192383  2315.480957   
                     2025-08-31 00:10:00  2247.658936  2312.984375   
                     2025-08-31 00:15:00  2246.772949  2311.389648   
                     2025-08-31 00:20:00  2245.689941  2309.847656   

                  

# FSWL

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from tabicl import TabICLForecaster
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\tabicl_5min_engineered_LASSO"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

BASE_WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288
CONTEXT_LEN = 288 * 30

LASSO_COEF_THRESHOLD = 1e-6

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample("5min").mean()
weather_5min = weather_df.resample("5min").mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# ENGINEERED FEATURES
# ============================================================
def add_engineered_features(data, target_col):
    data = data.copy()
    ts = data["timestamp"]

    data["hour"] = ts.dt.hour
    data["minute"] = ts.dt.minute
    data["dayofweek"] = ts.dt.dayofweek
    data["dayofyear"] = ts.dt.dayofyear
    data["month"] = ts.dt.month
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

    steps_per_day = 288
    steps_per_week = 288 * 7

    step_of_day = data["hour"] * 12 + data["minute"] // 5
    step_of_week = data["dayofweek"] * steps_per_day + step_of_day

    for k in [1, 2, 3]:
        data[f"sin_day_{k}"] = np.sin(2 * np.pi * k * step_of_day / steps_per_day)
        data[f"cos_day_{k}"] = np.cos(2 * np.pi * k * step_of_day / steps_per_day)

    for k in [1, 2]:
        data[f"sin_week_{k}"] = np.sin(2 * np.pi * k * step_of_week / steps_per_week)
        data[f"cos_week_{k}"] = np.cos(2 * np.pi * k * step_of_week / steps_per_week)

    lag_steps = [1, 2, 3, 6, 12, 24, 288, 576, 2016]

    for lag in lag_steps:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    rolling_windows = [3, 12, 288, 2016]
    shifted = data[target_col].shift(1)

    for window in rolling_windows:
        data[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
        data[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()

    data["temp_sq"] = data["Temp__C"] ** 2
    data["heating_degree_18"] = np.maximum(18 - data["Temp__C"], 0)
    data["cooling_degree_22"] = np.maximum(data["Temp__C"] - 22, 0)
    data["is_daylight"] = (data["SolarRad_W_m_2"] > 10).astype(int)

    return data


df = add_engineered_features(df, TARGET_COL)

ENGINEERED_COLS = [
    "hour", "minute", "dayofweek", "dayofyear", "month", "is_weekend",

    "sin_day_1", "cos_day_1",
    "sin_day_2", "cos_day_2",
    "sin_day_3", "cos_day_3",

    "sin_week_1", "cos_week_1",
    "sin_week_2", "cos_week_2",

    f"{TARGET_COL}_lag_1",
    f"{TARGET_COL}_lag_2",
    f"{TARGET_COL}_lag_3",
    f"{TARGET_COL}_lag_6",
    f"{TARGET_COL}_lag_12",
    f"{TARGET_COL}_lag_24",
    f"{TARGET_COL}_lag_288",
    f"{TARGET_COL}_lag_576",
    f"{TARGET_COL}_lag_2016",

    f"{TARGET_COL}_roll_mean_3",
    f"{TARGET_COL}_roll_std_3",
    f"{TARGET_COL}_roll_mean_12",
    f"{TARGET_COL}_roll_std_12",
    f"{TARGET_COL}_roll_mean_288",
    f"{TARGET_COL}_roll_std_288",
    f"{TARGET_COL}_roll_mean_2016",
    f"{TARGET_COL}_roll_std_2016",

    "temp_sq",
    "heating_degree_18",
    "cooling_degree_22",
    "is_daylight",
]

ALL_FEATURE_COLS = BASE_WEATHER_COLS + ENGINEERED_COLS

df = df.dropna(subset=[TARGET_COL] + ALL_FEATURE_COLS).reset_index(drop=True)

print("Total candidate features:", len(ALL_FEATURE_COLS))
print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# TABICL
# ============================================================
forecaster = TabICLForecaster()

# ============================================================
# LASSO FEATURE SELECTION
# ============================================================
def select_lasso_features(train_df, feature_cols, target_col, season, day_name):
    X = train_df[feature_cols].copy()
    y = train_df[target_col].copy()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LassoCV(
            alphas=np.logspace(-4, 2, 40),
            cv=5,
            max_iter=20000,
            random_state=42,
            n_jobs=-1,
        ))
    ])

    model.fit(X, y)

    lasso = model.named_steps["lasso"]
    coefs = lasso.coef_

    coef_df = pd.DataFrame({
        "feature": feature_cols,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs),
    })

    coef_df = coef_df.sort_values(
        "abs_coefficient", ascending=False
    ).reset_index(drop=True)

    selected_features = coef_df.loc[
        coef_df["abs_coefficient"] > LASSO_COEF_THRESHOLD,
        "feature"
    ].tolist()

    if len(selected_features) == 0:
        selected_features = coef_df.head(1)["feature"].tolist()

    coef_df["selected"] = coef_df["feature"].isin(selected_features)
    coef_df["season"] = season
    coef_df["day"] = day_name
    coef_df["lasso_alpha"] = lasso.alpha_

    return selected_features, coef_df

# ============================================================
# HELPER: FIND PREDICTION COLUMN
# ============================================================
def find_prediction_column(pred_df):
    possible_cols = [
        "prediction",
        "predictions",
        "mean",
        "median",
        "0.5",
        0.5,
        "target",
    ]

    for col in possible_cols:
        if col in pred_df.columns:
            return col

    numeric_cols = pred_df.select_dtypes(include=[np.number]).columns.tolist()

    numeric_cols = [
        c for c in numeric_cols
        if c not in ["timestamp"]
    ]

    if len(numeric_cols) > 0:
        return numeric_cols[0]

    raise ValueError(
        f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}"
    )

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    train_for_selection_df = df[df["timestamp"] < test_start].copy()

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(train_for_selection_df) == 0:
        print(f"Skipping {season}-{day_name}: no training data before {test_start}")
        return None, None, None

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None, None, None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None, None, None

    selected_features, coef_df = select_lasso_features(
        train_df=train_for_selection_df,
        feature_cols=ALL_FEATURE_COLS,
        target_col=TARGET_COL,
        season=season,
        day_name=day_name,
    )

    print(f"Selected {len(selected_features)} / {len(ALL_FEATURE_COLS)} features")
    print(selected_features)

    tabicl_df = history_df[["timestamp", TARGET_COL] + selected_features].copy()
    tabicl_df = tabicl_df.rename(columns={TARGET_COL: "target"})

    tabicl_df["item_id"] = "building_consumption"

    tabicl_df = tabicl_df[
        ["item_id", "timestamp", "target"] + selected_features
    ]

    # No future_df is passed.
    # TabiCL receives only historical target + LASSO-selected historical features.
    pred_df = forecaster.predict_df(
        tabicl_df,
        prediction_length=PRED_LEN,
    )

    print(pred_df.head())
    print(pred_df.columns.tolist())

    pred_col = find_prediction_column(pred_df)
    pred_values = pred_df[pred_col].values[:PRED_LEN]

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name
    out["n_selected_features"] = len(selected_features)

    selected_features_df = pd.DataFrame({
        "season": season,
        "day": day_name,
        "selected_feature": selected_features,
    })

    return out, coef_df, selected_features_df

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []
all_lasso_coefs = []
all_selected_features = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running TabiCL LASSO - {season} - {day_name}: {test_start}")

        res, coef_df, selected_features_df = run_one_forecast(
            season, day_name, test_start
        )

        if res is not None:
            all_results.append(res)
            all_lasso_coefs.append(coef_df)
            all_selected_features.append(selected_features_df)

            day_path = OUTPUT_DIR / f"tabicl_5min_FSWL_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            coef_path = OUTPUT_DIR / f"LASSO_coefficients_{season}_{day_name}.csv"
            coef_df.to_csv(coef_path, index=False)

            selected_path = OUTPUT_DIR / f"LASSO_selected_features_{season}_{day_name}.csv"
            selected_features_df.to_csv(selected_path, index=False)

            print(f"Saved prediction: {day_path}")
            print(f"Saved coefficients: {coef_path}")
            print(f"Saved selected features: {selected_path}")

# ============================================================
# SAVE COMBINED FILES
# ============================================================
if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

results_df = pd.concat(all_results, ignore_index=True)
lasso_coefs_all_df = pd.concat(all_lasso_coefs, ignore_index=True)
selected_all_df = pd.concat(all_selected_features, ignore_index=True)

results_df.to_csv(
    OUTPUT_DIR / "tabicl_5min_LASSO_ALL_predictions.csv",
    index=False
)

lasso_coefs_all_df.to_csv(
    OUTPUT_DIR / "LASSO_ALL_coefficients.csv",
    index=False
)

selected_all_df.to_csv(
    OUTPUT_DIR / "LASSO_ALL_selected_features.csv",
    index=False
)

print("Done.")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
print(f"Output folder: {OUTPUT_DIR}")

Total candidate features: 42
Data range: 2025-01-08 00:00:00 to 2025-12-31 23:55:00
Running TabiCL LASSO - winter - day1: 2025-01-31 00:00:00
Selected 30 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_lag_3', 'avg_power_roll_mean_12', 'cos_day_3', 'avg_power_roll_std_12', 'avg_power_roll_std_3', 'avg_power_lag_24', 'avg_power_lag_576', 'cos_day_1', 'avg_power_lag_12', 'avg_power_lag_288', 'avg_power_lag_2016', 'cos_day_2', 'sin_week_2', 'avg_power_roll_std_2016', 'sin_day_2', 'cos_week_1', 'sin_day_3', 'avg_power_roll_std_288', 'cos_week_2', 'avg_power_roll_mean_2016', 'Hum__', 'is_daylight', 'sin_day_1', 'hour', 'Rain_Mm', 'avg_power_roll_mean_288', 'sin_week_1', 'dayofyear']


Predicting time series: 100%|██████████| 1/1 [01:02<00:00, 62.12s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2893.047607  2382.549805   
                     2025-01-31 00:05:00  2885.459473  2379.413330   
                     2025-01-31 00:10:00  2883.682129  2377.165527   
                     2025-01-31 00:15:00  2887.363281  2373.707275   
                     2025-01-31 00:20:00  2888.847656  2370.393311   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2413.889893  2437.108643   
                     2025-01-31 00:05:00  2409.879395  2432.303711   
                     2025-01-31 00:10:00  2406.822021  2428.685547   
                     2025-01-31 00:15:00  2402.665771  2424.181152   
                     2025-01-31 00:20:00  2398.656738  2419.852539   

                  

Predicting time series: 100%|██████████| 1/1 [01:08<00:00, 68.64s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-02-15 00:00:00  2755.811279  2390.069824   
                     2025-02-15 00:05:00  2784.522949  2391.538574   
                     2025-02-15 00:10:00  2797.472412  2390.180908   
                     2025-02-15 00:15:00  2803.617188  2389.272949   
                     2025-02-15 00:20:00  2800.954834  2389.058594   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-02-15 00:00:00  2441.998047  2481.043213   
                     2025-02-15 00:05:00  2444.002930  2483.933594   
                     2025-02-15 00:10:00  2441.575928  2480.570801   
                     2025-02-15 00:15:00  2439.358643  2477.553223   
                     2025-02-15 00:20:00  2438.428711  2476.598633   

                  

...

                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  7357.890625  2722.282471   
                     2025-07-17 00:05:00  7373.949707  2695.538086   
                     2025-07-17 00:10:00  7427.758301  2645.926270   
                     2025-07-17 00:15:00  7508.464844  2589.122803   
                     2025-07-17 00:20:00  7603.824707  2530.756836   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  3777.355957  4428.418457   
                     2025-07-17 00:05:00  3757.740723  4421.666992   
                     2025-07-17 00:10:00  3721.362793  4408.664062   
                     2025-07-17 00:15:00  3680.141357  4396.294434   
                     2025-07-17 00:20:00  3635.571289  4385.552734   

                  

Predicting time series: 100%|██████████| 1/1 [01:30<00:00, 90.33s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  4543.926758  2149.033447   
                     2025-08-08 00:05:00  4549.727539  2149.500488   
                     2025-08-08 00:10:00  4545.260254  2144.426270   
                     2025-08-08 00:15:00  4544.396484  2139.302734   
                     2025-08-08 00:20:00  4537.032715  2137.175049   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  2593.388916  2958.543701   
                     2025-08-08 00:05:00  2592.690918  2957.341797   
                     2025-08-08 00:10:00  2586.341309  2951.797119   
                     2025-08-08 00:15:00  2580.990967  2947.669189   
                     2025-08-08 00:20:00  2577.760498  2944.395264   

                  

Predicting time series: 100%|██████████| 1/1 [01:18<00:00, 78.62s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2749.469482  2154.293945   
                     2025-08-31 00:05:00  2746.794189  2152.883789   
                     2025-08-31 00:10:00  2744.793213  2151.238281   
                     2025-08-31 00:15:00  2744.500732  2149.646729   
                     2025-08-31 00:20:00  2744.509766  2147.560547   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2252.395508  2318.013672   
                     2025-08-31 00:05:00  2250.216064  2314.791016   
                     2025-08-31 00:10:00  2248.364258  2312.130127   
                     2025-08-31 00:15:00  2246.962158  2310.173828   
                     2025-08-31 00:20:00  2245.393311  2308.328125   

                  

# Univariate

In [6]:
import pandas as pd
from pathlib import Path
import json
import time

from tabicl import TabICLForecaster

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\tabicl_5min_univariate"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

PRED_LEN = 288
CONTEXT_LEN = 288 * 30

# ============================================================
# LOAD LOAD DATA ONLY
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
load_df.columns = load_df.columns.str.strip()

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
load_df = load_df.set_index("timestamp")

load_5min = load_df.resample("5min").mean()

df = load_5min.reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())
print(df.head())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# LOAD TABICL
# ============================================================
forecaster = TabICLForecaster()

# ============================================================
# HELPER
# ============================================================
def find_prediction_column(pred_df):
    possible_cols = [
        "prediction",
        "predictions",
        "mean",
        "median",
        "0.5",
        0.5,
        "target",
    ]

    for col in possible_cols:
        if col in pred_df.columns:
            return col

    numeric_cols = pred_df.select_dtypes(include=["number"]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["timestamp"]]

    if len(numeric_cols) > 0:
        return numeric_cols[0]

    raise ValueError(
        f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}"
    )

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None

    # Univariate TabiCL input: only item_id, timestamp, target
    tabicl_df = history_df[["timestamp", TARGET_COL]].copy()
    tabicl_df = tabicl_df.rename(columns={TARGET_COL: "target"})
    tabicl_df["item_id"] = "building_consumption"

    tabicl_df = tabicl_df[
        ["item_id", "timestamp", "target"]
    ]

    pred_df = forecaster.predict_df(
        tabicl_df,
        prediction_length=PRED_LEN,
    )

    print(pred_df.head())
    print(pred_df.columns.tolist())

    pred_col = find_prediction_column(pred_df)
    pred_values = pred_df[pred_col].values[:PRED_LEN]

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name

    return out

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running TabiCL univariate - {season} - {day_name}: {test_start}")

        res = run_one_forecast(season, day_name, test_start)

        if res is not None:
            all_results.append(res)

            day_path = OUTPUT_DIR / f"tabicl_5min_univariate_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            print(f"Saved prediction: {day_path}")

# ============================================================
# SAVE COMBINED FILE
# ============================================================
if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

results_df = pd.concat(all_results, ignore_index=True)

all_path = OUTPUT_DIR / "tabicl_5min_univariate_ALL_predictions.csv"
results_df.to_csv(all_path, index=False)

print("Done.")
print(f"Saved combined predictions: {all_path}")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")

Data range: 2025-01-01 00:00:00 to 2025-12-31 23:55:00
            timestamp  avg_power
0 2025-01-01 00:00:00  2954.2428
1 2025-01-01 00:05:00  2642.1732
2 2025-01-01 00:10:00  2646.2412
3 2025-01-01 00:15:00  2685.5580
4 2025-01-01 00:20:00  2665.8096
Running TabiCL univariate - winter - day1: 2025-01-31 00:00:00


Predicting time series: 100%|██████████| 1/1 [01:13<00:00, 73.24s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2861.745361  2394.557129   
                     2025-01-31 00:05:00  2856.743408  2390.896484   
                     2025-01-31 00:10:00  2858.552734  2388.301270   
                     2025-01-31 00:15:00  2865.779297  2384.484131   
                     2025-01-31 00:20:00  2869.226807  2380.619629   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-01-31 00:00:00  2423.349121  2449.710693   
                     2025-01-31 00:05:00  2419.278320  2444.763184   
                     2025-01-31 00:10:00  2416.208008  2441.010010   
                     2025-01-31 00:15:00  2411.957275  2436.359375   
                     2025-01-31 00:20:00  2407.667480  2431.542725   

                  

Predicting time series: 100%|██████████| 1/1 [01:14<00:00, 74.69s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-02-15 00:00:00  2716.723145  2404.788086   
                     2025-02-15 00:05:00  2742.757568  2406.550537   
                     2025-02-15 00:10:00  2750.479004  2405.694336   
                     2025-02-15 00:15:00  2752.008789  2405.422852   
                     2025-02-15 00:20:00  2745.955566  2405.620605   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-02-15 00:00:00  2453.348145  2489.738281   
                     2025-02-15 00:05:00  2455.647949  2494.060791   
                     2025-02-15 00:10:00  2453.764404  2492.215332   
                     2025-02-15 00:15:00  2452.158203  2490.521729   
                     2025-02-15 00:20:00  2451.211670  2489.775391   

                  

...

                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  7070.568848  2693.473389   
                     2025-07-17 00:05:00  7075.515625  2668.876953   
                     2025-07-17 00:10:00  7109.581055  2616.686279   
                     2025-07-17 00:15:00  7172.458008  2561.301758   
                     2025-07-17 00:20:00  7245.435547  2504.833496   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-07-17 00:00:00  3714.940674  4333.725586   
                     2025-07-17 00:05:00  3696.135254  4323.605957   
                     2025-07-17 00:10:00  3655.230957  4305.053223   
                     2025-07-17 00:15:00  3613.951172  4292.979492   
                     2025-07-17 00:20:00  3567.848389  4280.155762   

                  

Predicting time series: 100%|██████████| 1/1 [01:10<00:00, 70.05s/it]


                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  4727.821289  2339.640381   
                     2025-08-08 00:05:00  4749.052734  2337.790527   
                     2025-08-08 00:10:00  4765.924805  2328.271240   
                     2025-08-08 00:15:00  4780.427246  2318.838623   
                     2025-08-08 00:20:00  4784.360352  2311.873047   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-08 00:00:00  2773.370850  3139.122070   
                     2025-08-08 00:05:00  2773.989258  3143.386963   
                     2025-08-08 00:10:00  2768.818115  3144.592285   
                     2025-08-08 00:15:00  2764.121338  3145.680664   
                     2025-08-08 00:20:00  2760.225098  3145.424805   

                  

Predicting time series: 100%|██████████| 1/1 [01:07<00:00, 67.93s/it]

                                               target          0.1  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2776.176270  2131.370361   
                     2025-08-31 00:05:00  2774.843506  2130.600098   
                     2025-08-31 00:10:00  2774.233398  2129.998047   
                     2025-08-31 00:15:00  2775.629395  2129.423096   
                     2025-08-31 00:20:00  2776.739990  2128.511963   

                                                  0.2          0.3  \
item_id              timestamp                                       
building_consumption 2025-08-31 00:00:00  2232.990723  2305.945801   
                     2025-08-31 00:05:00  2231.444580  2303.327637   
                     2025-08-31 00:10:00  2230.524170  2301.523682   
                     2025-08-31 00:15:00  2230.073975  2300.483398   
                     2025-08-31 00:20:00  2229.448486  2299.500977   

                  

# end